# 🚀 Proyecto Final: Sistema de Gestión de Paquetería Inteligente


## 🧩 Descripción General
Se debe implementar un sistema de gestión de envíos para una empresa de mensajería y logística que permita registrar paquetes, asociarlos a clientes, llevar el control de su estado, y calcular costos según el tipo de envío. El sistema debe incluir relaciones entre clases (composición, agregación, asociación), uso de métodos mágicos, herencia, encapsulación, y polimorfismo.

### 🎯 Objetivo del Proyecto
- Evaluar los siguientes conceptos:

- Clases y objetos

- Atributos públicos, protegidos y privados

- Métodos de instancia, clase y estáticos

- Métodos mágicos (__str__, __init__, etc.)

- Encapsulación con validaciones (getters/setters)

- Asociación, agregación y composición

- Herencia y polimorfismo

- Buenas prácticas (modularidad, documentación)

### ESTRUCTURA E LAS CLASES

| Clase           | Rol                                                                 |
|----------------|----------------------------------------------------------------------|
| Cliente         | Representa a un usuario que envía o recibe paquetes                  |
| Paquete         | Clase base para paquetes, incluye atributos comunes y métodos mágicos |
| PaqueteExpress  | Hereda de Paquete, incluye recargo por urgencia                      |
| PaqueteEstandar | Hereda de Paquete, sin recargo                                       |
| Direccion       | Composición con Cliente (cada cliente tiene una dirección)           |
| Envio           | Agrega un paquete y se asocia a un cliente                           |
| SistemaEnvios   | Administra clientes y envíos                                         |


### 🧪 Requisitos funcionales
1. Registrar clientes y asociarles una dirección.

2. Crear paquetes (express o estándar).

3. Asignar paquetes a envíos y clientes.

4. Calcular el precio de cada envío.

5. Mostrar listado de envíos con información detallada (polimorfismo).

6. Validar datos (precio, peso, nombre del cliente, etc.).

7. Mostrar reporte de todos los envíos realizados.

8. Agregar seguimiento del paquete (pendiente, en tránsito, entregado) usando métodos set_estado().

9. Guardar la información en un archivo .txt o .json.

10. Mostrar totales de ventas por tipo de envío (estándar vs express).

11. Diseñar un menú de opciones para registrar clientes/envíos de forma interactiva.



# Solucion

In [ ]:
import json
from abc import ABC, abstractmethod
from datetime import datetime

# ---------------------------- CLASES BASE ----------------------------
class Direccion:
    """Clase para representar direcciones (composición con Cliente)"""
    def __init__(self, calle: str, ciudad: str, codigo_postal: str, pais: str):
        self.calle = calle
        self.ciudad = ciudad
        self.codigo_postal = codigo_postal
        self.pais = pais
    
    def __str__(self):
        return f"{self.calle}, {self.ciudad}, {self.codigo_postal}, {self.pais}"

class Cliente:
    """Clase para representar clientes que envían/reciben paquetes"""
    def __init__(self, nombre: str, email: str, direccion: Direccion):
        self.nombre = nombre
        self.email = email
        self.direccion = direccion  # Composición
        self.envios = []  # Asociación con Envio
    
    def agregar_envio(self, envio):
        self.envios.append(envio)
    
    def __str__(self):
        return f"Cliente: {self.nombre} | Email: {self.email} | Dirección: {self.direccion}"

class Paquete(ABC):
    """Clase abstracta base para paquetes"""
    def __init__(self, peso: float, dimensiones: tuple, estado: str = "pendiente"):
        self._peso = peso
        self._dimensiones = dimensiones
        self._estado = estado
        self._fecha_registro = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    @property
    def peso(self):
        return self._peso
    
    @peso.setter
    def peso(self, valor):
        if valor <= 0:
            raise ValueError("El peso debe ser mayor a cero")
        self._peso = valor
    
    @property
    def dimensiones(self):
        return self._dimensiones
    
    @dimensiones.setter
    def dimensiones(self, valor):
        if any(dim <= 0 for dim in valor):
            raise ValueError("Todas las dimensiones deben ser positivas")
        self._dimensiones = valor
    
    @property
    def estado(self):
        return self._estado
    
    def set_estado(self, nuevo_estado: str):
        estados_validos = ["pendiente", "en tránsito", "entregado"]
        if nuevo_estado.lower() not in estados_validos:
            raise ValueError(f"Estado inválido. Debe ser uno de: {', '.join(estados_validos)}")
        self._estado = nuevo_estado.lower()
    
    @abstractmethod
    def calcular_costo(self):
        pass
    
    def __str__(self):
        return (f"Paquete: {self.__class__.__name__} | Peso: {self._peso} kg | "
                f"Dimensiones: {self._dimensiones} | Estado: {self._estado}")

class PaqueteEstandar(Paquete):
    """Paquete estándar sin recargo"""
    def calcular_costo(self):
        # Costo base: $5 por kg + $2 por cada dimensión (largo, ancho, alto)
        return 5 * self.peso + 2 * sum(self.dimensiones)
    
    def __str__(self):
        return super().__str__() + f" | Costo: ${self.calcular_costo():.2f}"

class PaqueteExpress(Paquete):
    """Paquete express con recargo por urgencia"""
    def __init__(self, peso: float, dimensiones: tuple, urgencia: int = 1):
        super().__init__(peso, dimensiones)
        self.urgencia = urgencia  # 1: normal, 2: urgente, 3: muy urgente
    
    @property
    def urgencia(self):
        return self._urgencia
    
    @urgencia.setter
    def urgencia(self, valor):
        if valor not in (1, 2, 3):
            raise ValueError("Urgencia debe ser 1 (normal), 2 (urgente) o 3 (muy urgente)")
        self._urgencia = valor
    
    def calcular_costo(self):
        # Costo base como estándar + recargo por urgencia (20%, 50%, 100%)
        costo_base = 5 * self.peso + 2 * sum(self.dimensiones)
        recargos = {1: 0.2, 2: 0.5, 3: 1.0}
        return costo_base * (1 + recargos[self.urgencia])
    
    def __str__(self):
        return (super().__str__() + 
                f" | Urgencia: {self.urgencia} | Costo: ${self.calcular_costo():.2f}")

class Envio:
    """Clase que agrega un paquete y se asocia a un cliente"""
    def __init__(self, cliente: Cliente, paquete: Paquete, destino: Direccion):
        self.cliente = cliente
        self.paquete = paquete  # Agregación
        self.destino = destino
        self.fecha_envio = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        cliente.agregar_envio(self)  # Asociación bidireccional
    
    def __str__(self):
        return (f"Envio para {self.cliente.nombre} | "
                f"Paquete: {self.paquete.__class__.__name__} | "
                f"Destino: {self.destino} | Fecha: {self.fecha_envio}")

class SistemaEnvios:
    """Sistema principal que gestiona clientes y envíos"""
    def __init__(self):
        self.clientes = []
        self.envios = []
    
    def registrar_cliente(self, nombre: str, email: str, direccion: Direccion):
        # Validar email simple
        if "@" not in email or "." not in email:
            raise ValueError("Email inválido")
        
        cliente = Cliente(nombre, email, direccion)
        self.clientes.append(cliente)
        return cliente
    
    def crear_paquete(self, tipo: str, peso: float, dimensiones: tuple, **kwargs):
        if tipo.lower() == "estandar":
            return PaqueteEstandar(peso, dimensiones)
        elif tipo.lower() == "express":
            return PaqueteExpress(peso, dimensiones, kwargs.get('urgencia', 1))
        else:
            raise ValueError("Tipo de paquete inválido. Debe ser 'estandar' o 'express'")
    
    def crear_envio(self, cliente: Cliente, paquete: Paquete, destino: Direccion):
        envio = Envio(cliente, paquete, destino)
        self.envios.append(envio)
        return envio
    
    def mostrar_envios(self):
        for envio in self.envios:
            print(envio)
            print(f"  - {envio.paquete}")
            print("-" * 50)
    
    def total_ventas(self):
        total_estandar = sum(e.paquete.calcular_costo() 
                            for e in self.envios 
                            if isinstance(e.paquete, PaqueteEstandar))
        total_express = sum(e.paquete.calcular_costo() 
                           for e in self.envios 
                           if isinstance(e.paquete, PaqueteExpress))
        
        print("\nREPORTE DE VENTAS")
        print(f"Total envíos estándar: ${total_estandar:.2f}")
        print(f"Total envíos express: ${total_express:.2f}")
        print(f"TOTAL GENERAL: ${total_estandar + total_express:.2f}")
    
    def guardar_datos(self, archivo: str = "envios.json"):
        datos = {
            "clientes": [
                {
                    "nombre": c.nombre,
                    "email": c.email,
                    "direccion": {
                        "calle": c.direccion.calle,
                        "ciudad": c.direccion.ciudad,
                        "codigo_postal": c.direccion.codigo_postal,
                        "pais": c.direccion.pais
                    }
                } for c in self.clientes
            ],
            "envios": [
                {
                    "cliente": e.cliente.nombre,
                    "paquete": {
                        "tipo": e.paquete.__class__.__name__,
                        "peso": e.paquete.peso,
                        "dimensiones": e.paquete.dimensiones,
                        "estado": e.paquete.estado,
                        **({"urgencia": e.paquete.urgencia} 
                           if isinstance(e.paquete, PaqueteExpress) else {})
                    },
                    "destino": {
                        "calle": e.destino.calle,
                        "ciudad": e.destino.ciudad,
                        "codigo_postal": e.destino.codigo_postal,
                        "pais": e.destino.pais
                    },
                    "fecha_envio": e.fecha_envio
                } for e in self.envios
            ]
        }
        
        with open(archivo, "w") as f:
            json.dump(datos, f, indent=4)
    
    def cargar_datos(self, archivo: str = "envios.json"):
        try:
            with open(archivo, "r") as f:
                datos = json.load(f)
                
                # Limpiar datos actuales
                self.clientes = []
                self.envios = []
                
                # Cargar clientes
                clientes_map = {}
                for c_data in datos["clientes"]:
                    direccion = Direccion(
                        c_data["direccion"]["calle"],
                        c_data["direccion"]["ciudad"],
                        c_data["direccion"]["codigo_postal"],
                        c_data["direccion"]["pais"]
                    )
                    cliente = self.registrar_cliente(
                        c_data["nombre"],
                        c_data["email"],
                        direccion
                    )
                    clientes_map[c_data["nombre"]] = cliente
                
                # Cargar envíos
                for e_data in datos["envios"]:
                    cliente = clientes_map[e_data["cliente"]]
                    
                    # Crear paquete
                    if e_data["paquete"]["tipo"] == "PaqueteEstandar":
                        paquete = self.crear_paquete(
                            "estandar",
                            e_data["paquete"]["peso"],
                            tuple(e_data["paquete"]["dimensiones"])
                        )
                    else:
                        paquete = self.crear_paquete(
                            "express",
                            e_data["paquete"]["peso"],
                            tuple(e_data["paquete"]["dimensiones"]),
                            urgencia=e_data["paquete"]["urgencia"]
                        )
                    
                    paquete.set_estado(e_data["paquete"]["estado"])
                    
                    # Crear destino
                    destino = Direccion(
                        e_data["destino"]["calle"],
                        e_data["destino"]["ciudad"],
                        e_data["destino"]["codigo_postal"],
                        e_data["destino"]["pais"]
                    )
                    
                    # Crear envío (la fecha se sobreescribe con la del archivo)
                    envio = self.crear_envio(cliente, paquete, destino)
                    envio.fecha_envio = e_data["fecha_envio"]
                    
        except FileNotFoundError:
            print(f"Archivo {archivo} no encontrado. Se creará uno nuevo al guardar.")
        except Exception as e:
            print(f"Error al cargar datos: {str(e)}")

# ---------------------------- MENU INTERACTIVO ----------------------------
def menu_principal():
    sistema = SistemaEnvios()
    sistema.cargar_datos()  # Intentar cargar datos existentes
    
    while True:
        print("\n📦 SISTEMA DE GESTIÓN DE PAQUETERÍA 📦")
        print("1. Registrar cliente")
        print("2. Crear envío")
        print("3. Mostrar todos los envíos")
        print("4. Generar reporte de ventas")
        print("5. Cambiar estado de un paquete")
        print("6. Guardar datos")
        print("7. Salir")
        
        opcion = input("Seleccione una opción: ")
        
        if opcion == "1":
            print("\nREGISTRAR NUEVO CLIENTE")
            nombre = input("Nombre completo: ")
            email = input("Email: ")
            
            print("\nDirección del cliente:")
            calle = input("Calle: ")
            ciudad = input("Ciudad: ")
            codigo_postal = input("Código postal: ")
            pais = input("País: ")
            
            try:
                direccion = Direccion(calle, ciudad, codigo_postal, pais)
                cliente = sistema.registrar_cliente(nombre, email, direccion)
                print(f"\n✅ Cliente registrado: {cliente}")
            except Exception as e:
                print(f"\n❌ Error: {str(e)}")
        
        elif opcion == "2":
            if not sistema.clientes:
                print("\n❌ No hay clientes registrados. Registre uno primero.")
                continue
            
            print("\nCREAR NUEVO ENVÍO")
            print("Clientes disponibles:")
            for i, cliente in enumerate(sistema.clientes, 1):
                print(f"{i}. {cliente.nombre}")
            
            try:
                cliente_idx = int(input("Seleccione cliente (número): ")) - 1
                cliente = sistema.clientes[cliente_idx]
                
                tipo = input("Tipo de paquete (estandar/express): ").lower()
                peso = float(input("Peso (kg): "))
                dimensiones = tuple(map(float, input("Dimensiones (largo,ancho,alto separados por comas): ").split(",")))
                
                if tipo == "express":
                    urgencia = int(input("Nivel de urgencia (1:normal, 2:urgente, 3:muy urgente): "))
                    paquete = sistema.crear_paquete(tipo, peso, dimensiones, urgencia=urgencia)
                else:
                    paquete = sistema.crear_paquete(tipo, peso, dimensiones)
                
                print("\nDirección de destino:")
                calle = input("Calle: ")
                ciudad = input("Ciudad: ")
                codigo_postal = input("Código postal: ")
                pais = input("País: ")
                
                destino = Direccion(calle, ciudad, codigo_postal, pais)
                envio = sistema.crear_envio(cliente, paquete, destino)
                
                print(f"\n✅ Envío creado:\n{envio}\n{paquete}")
            except Exception as e:
                print(f"\n❌ Error: {str(e)}")
        
        elif opcion == "3":
            print("\nLISTADO DE ENVÍOS")
            sistema.mostrar_envios()
        
        elif opcion == "4":
            sistema.total_ventas()
        
        elif opcion == "5":
            if not sistema.envios:
                print("\n❌ No hay envíos registrados.")
                continue
            
            print("\nCAMBIAR ESTADO DE PAQUETE")
            for i, envio in enumerate(sistema.envios, 1):
                print(f"{i}. {envio.cliente.nombre} - {envio.paquete.estado}")
            
            try:
                envio_idx = int(input("Seleccione envío (número): ")) - 1
                envio = sistema.envios[envio_idx]
                
                print(f"\nPaquete seleccionado: {envio.paquete}")
                nuevo_estado = input("Nuevo estado (pendiente/en tránsito/entregado): ")
                envio.paquete.set_estado(nuevo_estado)
                print(f"\n✅ Estado actualizado: {envio.paquete.estado}")
            except Exception as e:
                print(f"\n❌ Error: {str(e)}")
        
        elif opcion == "6":
            sistema.guardar_datos()
            print("\n✅ Datos guardados correctamente en 'envios.json'")
        
        elif opcion == "7":
            print("\n¡Gracias por usar el sistema de gestión de paquetería!")
            break
        
        else:
            print("\n❌ Opción inválida. Intente nuevamente.")

# Ejecutar el sistema
if __name__ == "__main__":
    menu_principal()

Archivo envios.json no encontrado. Se creará uno nuevo al guardar.

📦 SISTEMA DE GESTIÓN DE PAQUETERÍA 📦
1. Registrar cliente
2. Crear envío
3. Mostrar todos los envíos
4. Generar reporte de ventas
5. Cambiar estado de un paquete
6. Guardar datos
7. Salir
